# Whisper-KDA vs Whisper-tiny: speed benchmark

Ноутбук сравнивает вычислительную скорость двух моделей на NVIDIA T4:

1. `teacher_forced_forward` — полный encoder + decoder forward при одинаковой длине decoder sequence;
2. `greedy_generate_cache` — end-to-end greedy generation с cache.

Обе модели получают один и тот же тензор Whisper log-Mel features и имеют случайно инициализированные веса. Общий Whisper encoder и token embeddings создаются один раз и точно копируются в Whisper-KDA. Для остальных decoder weights используются одинаковый seed, hidden size, число слоёв/голов и размер vocabulary; все decoder tensors не могут быть одинаковыми, потому что KDA и Transformer содержат разные операции и формы параметров. Перед измерениями выполняется warm-up и синхронизация CUDA.

> Это benchmark производительности, а не качества распознавания: случайно инициализированные модели не должны использоваться для сравнения WER.

In [3]:
%pip install -q "transformers==5.13.1" "flash-linear-attention[cuda]==0.5.2" pandas matplotlib

In [4]:
%cd /content

!git clone https://github.com/SummerSchoolSPBU2026/whisper-kda-asr.git
%cd /content/whisper-kda-asr

/content
Cloning into 'whisper-kda-asr'...
remote: Enumerating objects: 51, done.
remote: Counting objects: 100% (51/51), done.
remote: Compressing objects: 100% (34/34), done.
remote: Total 51 (delta 20), reused 45 (delta 14), pack-reused 0 (from 0)
Receiving objects: 100% (51/51), 657.24 KiB | 28.58 MiB/s, done.
Resolving deltas: 100% (20/20), done.
/content/whisper-kda-asr


In [5]:
import gc
import math
import statistics
import subprocess
import threading
import time

import matplotlib.pyplot as plt
import pandas as pd
import torch
from transformers import (
    WhisperConfig,
    WhisperForConditionalGeneration,
)

from model import WhisperKDAModel, create_kda_config

torch.manual_seed(42)

if not torch.cuda.is_available():
    raise RuntimeError("Для benchmark нужна CUDA GPU")

device = torch.device("cuda")
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(device))

ModuleNotFoundError: No module named 'model.utils'; 'model' is not a package

In [ ]:
WHISPER_MODEL_NAME = "openai/whisper-tiny"
RANDOM_SEED = 42

whisper_config = WhisperConfig.from_pretrained(WHISPER_MODEL_NAME)
whisper_config._attn_implementation = "sdpa"

class BenchmarkVocabulary:
    pad_token_id = 0
    bos_token_id = whisper_config.decoder_start_token_id
    eos_token_id = whisper_config.eos_token_id

    def __len__(self):
        return whisper_config.vocab_size

benchmark_vocabulary = BenchmarkVocabulary()
assert len({
    benchmark_vocabulary.pad_token_id,
    benchmark_vocabulary.bos_token_id,
    benchmark_vocabulary.eos_token_id,
}) == 3

torch.manual_seed(RANDOM_SEED)
whisper_model = WhisperForConditionalGeneration(whisper_config).eval()
whisper_model.model.encoder.requires_grad_(False)

print("Shared vocabulary size:", len(benchmark_vocabulary))

## Параметры

`TEXT_LENGTH` задаёт одинаковую teacher-forced длину decoder для обеих моделей. `MAX_NEW_TOKENS` ограничивает greedy generation.


In [ ]:
BATCH_SIZE = 1
TEXT_LENGTH = 64
MAX_NEW_TOKENS = 32
WARMUP_RUNS = 1
MEASURED_RUNS = 5
AUDIO_SECONDS = 30.0
RESOURCE_PROFILE_SECONDS = 3.0
RESOURCE_SAMPLE_INTERVAL_SECONDS = 0.1

# Whisper принимает 80 mel bins и 3000 frames для окна до 30 секунд.
input_features = torch.randn(
    BATCH_SIZE,
    80,
    3000,
    dtype=torch.float32,
    device=device,
)
audio_attention_mask = torch.ones(
    BATCH_SIZE,
    3000,
    dtype=torch.long,
    device=device,
)

print("input_features:", tuple(input_features.shape))

In [ ]:
def synchronize():
    torch.cuda.synchronize(device)


def percentile(values, fraction):
    ordered = sorted(values)
    index = min(len(ordered) - 1, math.ceil(fraction * len(ordered)) - 1)
    return ordered[index]


def parameter_count(model):
    return sum(parameter.numel() for parameter in model.parameters())


def trainable_parameter_count(model):
    return sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )


def parameter_memory_mb(model):
    return sum(
        parameter.numel() * parameter.element_size()
        for parameter in model.parameters()
    ) / 1024**2


def benchmark_call(call, warmup_runs=WARMUP_RUNS, measured_runs=MEASURED_RUNS):
    synchronize()
    warmup_start = time.perf_counter()
    for _ in range(warmup_runs):
        call()
    synchronize()
    warmup_seconds = time.perf_counter() - warmup_start

    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats(device)

    timings = []
    for _ in range(measured_runs):
        synchronize()
        start = time.perf_counter()
        call()
        synchronize()
        timings.append(time.perf_counter() - start)

    return {
        "warmup_seconds": warmup_seconds,
        "mean_ms": statistics.mean(timings) * 1000,
        "median_ms": statistics.median(timings) * 1000,
        "p90_ms": percentile(timings, 0.90) * 1000,
        "min_ms": min(timings) * 1000,
        "peak_memory_mb": torch.cuda.max_memory_allocated(device) / 1024**2,
        "peak_reserved_mb": torch.cuda.max_memory_reserved(device) / 1024**2,
    }


class NvidiaSmiSampler:
    def __init__(self, interval_seconds=0.1):
        self.interval_seconds = interval_seconds
        self.samples = []
        self.stop_event = threading.Event()
        self.thread = None

    def _sample(self):
        completed = subprocess.run(
            [
                "nvidia-smi",
                "-i",
                str(torch.cuda.current_device()),
                "--query-gpu=power.draw,utilization.gpu,memory.used",
                "--format=csv,noheader,nounits",
            ],
            check=True,
            capture_output=True,
            text=True,
        )
        power_w, utilization_pct, memory_mb = [
            float(value.strip())
            for value in completed.stdout.strip().split(",")
        ]
        self.samples.append({
            "power_w": power_w,
            "utilization_pct": utilization_pct,
            "device_memory_mb": memory_mb,
        })

    def _run(self):
        while not self.stop_event.is_set():
            try:
                self._sample()
            except (FileNotFoundError, subprocess.SubprocessError, ValueError):
                pass
            self.stop_event.wait(self.interval_seconds)

    def start(self):
        self.thread = threading.Thread(target=self._run, daemon=True)
        self.thread.start()

    def stop(self):
        self.stop_event.set()
        self.thread.join()


def profile_resources(call, minimum_seconds=RESOURCE_PROFILE_SECONDS):
    sampler = NvidiaSmiSampler(RESOURCE_SAMPLE_INTERVAL_SECONDS)
    synchronize()
    sampler.start()
    start = time.perf_counter()
    calls = 0

    while time.perf_counter() - start < minimum_seconds:
        call()
        synchronize()
        calls += 1

    elapsed_seconds = time.perf_counter() - start
    sampler.stop()

    if not sampler.samples:
        return {
            "resource_profile_seconds": elapsed_seconds,
            "resource_profile_calls": calls,
            "avg_gpu_utilization_pct": None,
            "max_gpu_utilization_pct": None,
            "avg_power_w": None,
            "max_power_w": None,
            "estimated_energy_per_call_j": None,
            "peak_device_memory_mb": None,
        }

    average_power_w = statistics.mean(
        sample["power_w"] for sample in sampler.samples
    )

    return {
        "resource_profile_seconds": elapsed_seconds,
        "resource_profile_calls": calls,
        "avg_gpu_utilization_pct": statistics.mean(
            sample["utilization_pct"] for sample in sampler.samples
        ),
        "max_gpu_utilization_pct": max(
            sample["utilization_pct"] for sample in sampler.samples
        ),
        "avg_power_w": average_power_w,
        "max_power_w": max(sample["power_w"] for sample in sampler.samples),
        "estimated_energy_per_call_j": average_power_w * elapsed_seconds / calls,
        "peak_device_memory_mb": max(
            sample["device_memory_mb"] for sample in sampler.samples
        ),
    }


def release_cuda_memory():
    gc.collect()
    torch.cuda.empty_cache()
    synchronize()


results = []

## Whisper-KDA

Модель загружается и измеряется отдельно, чтобы веса второй модели не попадали в peak GPU memory.

In [ ]:
kda_config = create_kda_config(
    tokenizer=benchmark_vocabulary,
    max_target_length=max(TEXT_LENGTH, MAX_NEW_TOKENS + 1),
    freeze_encoder=True,
    random_init_encoder=True,
)

torch.manual_seed(RANDOM_SEED)
kda_model = WhisperKDAModel(config=kda_config)
kda_model.encoder.load_state_dict(
    whisper_model.model.encoder.state_dict()
)
with torch.no_grad():
    kda_model.decoder.model.embeddings.weight.copy_(
        whisper_model.model.decoder.embed_tokens.weight
    )

encoder_weights_equal = all(
    torch.equal(kda_parameter, whisper_parameter)
    for kda_parameter, whisper_parameter in zip(
        kda_model.encoder.parameters(),
        whisper_model.model.encoder.parameters(),
    )
)
assert encoder_weights_equal, "Encoder weights должны полностью совпадать"
assert torch.equal(
    kda_model.decoder.model.embeddings.weight,
    whisper_model.model.decoder.embed_tokens.weight,
), "Token embedding weights должны полностью совпадать"

load_start = time.perf_counter()
kda_model = kda_model.to(device).eval()
synchronize()
kda_load_seconds = time.perf_counter() - load_start

kda_decoder_ids = torch.full(
    (BATCH_SIZE, TEXT_LENGTH),
    kda_config.bos_token_id,
    dtype=torch.long,
    device=device,
)

@torch.inference_mode()
def kda_forward():
    with torch.autocast(device_type="cuda", dtype=torch.float16):
        output = kda_model(
            input_features=input_features,
            attention_mask=audio_attention_mask,
            decoder_input_ids=kda_decoder_ids,
            use_cache=False,
        )
        _ = output.logits[:, -1, 0]

kda_forward_result = benchmark_call(kda_forward)
kda_forward_result.update(profile_resources(kda_forward))
kda_forward_result.update({
    "model": "Whisper-KDA",
    "benchmark": "teacher_forced_forward",
    "load_seconds": kda_load_seconds,
    "parameters_m": parameter_count(kda_model) / 1e6,
    "trainable_parameters_m": trainable_parameter_count(kda_model) / 1e6,
    "parameter_memory_mb": parameter_memory_mb(kda_model),
    "processed_tokens": BATCH_SIZE * TEXT_LENGTH,
})
results.append(kda_forward_result)

kda_generated_tokens = 0

@torch.inference_mode()
def kda_generate():
    global kda_generated_tokens
    output_ids = kda_model.generate(
        input_features=input_features,
        attention_mask=audio_attention_mask,
        max_new_tokens=MAX_NEW_TOKENS,
        num_beams=1,
        do_sample=False,
        use_cache=True,
    )
    kda_generated_tokens = output_ids.size(1) - 1

kda_generation_result = benchmark_call(kda_generate)
kda_generation_result.update(profile_resources(kda_generate))
kda_generation_result.update({
    "model": "Whisper-KDA",
    "benchmark": "greedy_generate_cache",
    "load_seconds": kda_load_seconds,
    "parameters_m": parameter_count(kda_model) / 1e6,
    "trainable_parameters_m": trainable_parameter_count(kda_model) / 1e6,
    "parameter_memory_mb": parameter_memory_mb(kda_model),
    "processed_tokens": BATCH_SIZE * kda_generated_tokens,
})
results.append(kda_generation_result)

print("KDA load, s:", round(kda_load_seconds, 3))
print("KDA generated tokens per sample:", kda_generated_tokens)

del kda_model, kda_decoder_ids
release_cuda_memory()


## Обычный Whisper-tiny

In [ ]:
load_start = time.perf_counter()
whisper_model = whisper_model.to(device).eval()
synchronize()
whisper_load_seconds = time.perf_counter() - load_start

whisper_decoder_start = whisper_model.config.decoder_start_token_id
whisper_decoder_ids = torch.full(
    (BATCH_SIZE, TEXT_LENGTH),
    whisper_decoder_start,
    dtype=torch.long,
    device=device,
)

@torch.inference_mode()
def whisper_forward():
    with torch.autocast(device_type="cuda", dtype=torch.float16):
        output = whisper_model(
            input_features=input_features,
            attention_mask=audio_attention_mask,
            decoder_input_ids=whisper_decoder_ids,
            use_cache=False,
        )
        _ = output.logits[:, -1, 0]

whisper_forward_result = benchmark_call(whisper_forward)
whisper_forward_result.update(profile_resources(whisper_forward))
whisper_forward_result.update({
    "model": "Whisper-tiny",
    "benchmark": "teacher_forced_forward",
    "load_seconds": whisper_load_seconds,
    "parameters_m": parameter_count(whisper_model) / 1e6,
    "trainable_parameters_m": trainable_parameter_count(whisper_model) / 1e6,
    "parameter_memory_mb": parameter_memory_mb(whisper_model),
    "processed_tokens": BATCH_SIZE * TEXT_LENGTH,
})
results.append(whisper_forward_result)

whisper_prompt_ids = torch.full(
    (BATCH_SIZE, 1),
    whisper_decoder_start,
    dtype=torch.long,
    device=device,
)
whisper_generated_tokens = 0

@torch.inference_mode()
def whisper_generate():
    global whisper_generated_tokens
    output_ids = whisper_model.generate(
        input_features=input_features,
        attention_mask=audio_attention_mask,
        decoder_input_ids=whisper_prompt_ids,
        max_new_tokens=MAX_NEW_TOKENS,
        num_beams=1,
        do_sample=False,
        use_cache=True,
        forced_decoder_ids=None,
    )
    whisper_generated_tokens = output_ids.size(1) - whisper_prompt_ids.size(1)

whisper_generation_result = benchmark_call(whisper_generate)
whisper_generation_result.update(profile_resources(whisper_generate))
whisper_generation_result.update({
    "model": "Whisper-tiny",
    "benchmark": "greedy_generate_cache",
    "load_seconds": whisper_load_seconds,
    "parameters_m": parameter_count(whisper_model) / 1e6,
    "trainable_parameters_m": trainable_parameter_count(whisper_model) / 1e6,
    "parameter_memory_mb": parameter_memory_mb(whisper_model),
    "processed_tokens": BATCH_SIZE * whisper_generated_tokens,
})
results.append(whisper_generation_result)

print("Whisper load, s:", round(whisper_load_seconds, 3))
print("Whisper generated tokens per sample:", whisper_generated_tokens)

del whisper_model, whisper_decoder_ids, whisper_prompt_ids
release_cuda_memory()


## Результаты

`median_ms` — основной показатель latency после warm-up. `warmup_seconds` полезен отдельно: для Whisper-KDA туда входит первая компиляция Triton. `peak_memory_mb` и `peak_reserved_mb` берутся из PyTorch. Полная память устройства, загрузка и мощность GPU собираются отдельным проходом через `nvidia-smi`, поэтому sampling не искажает основной замер latency.

In [ ]:
results_df = pd.DataFrame(results)
results_df["tokens_per_second"] = (
    results_df["processed_tokens"] / (results_df["median_ms"] / 1000)
)
results_df["samples_per_second"] = BATCH_SIZE / (results_df["median_ms"] / 1000)
results_df["rtf"] = results_df["median_ms"] / 1000 / AUDIO_SECONDS

columns = [
    "model",
    "benchmark",
    "parameters_m",
    "trainable_parameters_m",
    "parameter_memory_mb",
    "load_seconds",
    "warmup_seconds",
    "median_ms",
    "mean_ms",
    "p90_ms",
    "tokens_per_second",
    "samples_per_second",
    "rtf",
    "peak_memory_mb",
    "peak_reserved_mb",
    "peak_device_memory_mb",
    "avg_gpu_utilization_pct",
    "max_gpu_utilization_pct",
    "avg_power_w",
    "max_power_w",
    "estimated_energy_per_call_j",
]

display(results_df[columns].round(3))

fig, axes = plt.subplots(1, 3, figsize=(17, 4))

for benchmark_name, group in results_df.groupby("benchmark"):
    axes[0].bar(
        [f"{model}\n{benchmark_name}" for model in group["model"]],
        group["median_ms"],
        label=benchmark_name,
    )
axes[0].set_title("Median latency after warm-up")
axes[0].set_ylabel("Milliseconds")
axes[0].tick_params(axis="x", rotation=15)

memory = results_df.groupby("model", as_index=False)["peak_memory_mb"].max()
axes[1].bar(memory["model"], memory["peak_memory_mb"])
axes[1].set_title("Peak allocated GPU memory")
axes[1].set_ylabel("MiB")

for benchmark_name, group in results_df.groupby("benchmark"):
    axes[2].bar(
        [f"{model}\n{benchmark_name}" for model in group["model"]],
        group["estimated_energy_per_call_j"],
        label=benchmark_name,
    )
axes[2].set_title("Estimated energy per call")
axes[2].set_ylabel("Joules")
axes[2].tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.show()

results_df.to_csv("whisper_kda_vs_whisper_tiny_benchmark.csv", index=False)
print("Saved: whisper_kda_vs_whisper_tiny_benchmark.csv")

## Как интерпретировать

- Для архитектурного сравнения в первую очередь используйте `teacher_forced_forward`: у обеих моделей одинаковые batch, audio shape и decoder length.
- Для пользовательской latency используйте `greedy_generate_cache`, но учитывайте фактическое число `processed_tokens`: случайные модели могут сгенерировать EOS на разных шагах.
- Не сравнивайте `warmup_seconds` с `median_ms`: warm-up Whisper-KDA включает разовую JIT-компиляцию Triton.
- `estimated_energy_per_call_j` — приближённая оценка: средняя мощность из `nvidia-smi`, умноженная на длительность ресурсного прохода и разделённая на число вызовов.
- `peak_memory_mb` показывает tensor allocations PyTorch, а `peak_device_memory_mb` — всю занятую память GPU, включая CUDA context и сторонние процессы.
- Повторяйте эксперимент на одной GPU без других процессов и указывайте версии PyTorch, CUDA, Transformers и FLA.
- Для отчёта запустите notebook минимум два раза в свежем runtime и сохраните оба CSV. Первый запуск дополнительно включает холодные загрузки и компиляцию.